# Evaluation attempt inspection

Load a single already-completed evaluation attempt and inspect:
the stored prompts, model response metrics, pass/fail card, and board plots.

**Option A — exact file (recommended):** set `FILE_PATH` to a path relative to the
project root, e.g.
`outputs/evaluation/2g_2b_all/runs/20260607T201639906293Z_sonnet-medium-low_e4eac55c/attempts/2g_2b_all.low.r00.g01.b00__anthropic_claude-sonnet-4-6__medium__forbidden-snippets.json`

**Option B — search by job ID:** leave `FILE_PATH = None` and set `JOB_ID` to the
filename stem (without `.json`). Optionally set `CASE_SET` to narrow the search when
the same job ID appears in multiple case sets.

The move plot is produced for one 2D plane per axis pair that includes the move axis.
A 4D move on axis 0 produces planes (0,1), (0,2), (0,3).  
Boards with more than 4 dimensions cannot be plotted.

In [9]:
import sys
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "visualization":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from visualization.evaluation_figures import (
    display_attempt_prompt,
    display_attempt_response,
    display_attempt_summary,
    load_evaluation_attempt,
    plot_attempt_move,
)

In [10]:
# Option A: relative path from project root — takes priority over JOB_ID when set
FILE_PATH = "outputs/evaluation/1g_1b_lmh/runs/20260614T154059113012Z_gpt5-vs-opus4-8_14fcbee8/attempts/1g_1b_lmh.low.r00.g00.b00__openai_gpt-5__high__forbidden-snippets.json"

# Option B: job ID search (used only when FILE_PATH is None)
JOB_ID = "2g_2b_all.low.r00.g01.b00__anthropic_claude-sonnet-4-6__medium__forbidden-snippets"
CASE_SET = None  # set to e.g. "2g_2b_all" to narrow the file search

In [11]:
if FILE_PATH is not None:
    context = load_evaluation_attempt(PROJECT_ROOT / FILE_PATH)
else:
    context = load_evaluation_attempt(JOB_ID, case_set=CASE_SET)

## Prompt

In [12]:
display(display_attempt_prompt(context))


### Attempt
`1g_1b_lmh.low.r00.g00.b00__openai_gpt-5__high__forbidden-snippets.json`

### System prompt
```text
You solve a multidimensional formal-language Scrabble benchmark.
Follow the supplied rules exactly and return only one JSON object with exactly the fields `start`, `axis`, and `sequence`.
Do not use tools or add explanations.
```

### User prompt (board configuration redacted)
```text
Place exactly one contiguous sequence. Maximize the number of newly placed rack symbols.

## Move geometry
- Coordinates are zero-based vectors with 2 entries.
- `start` is the first coordinate. `axis` advances one coordinate component per symbol.
- Axis selects the board dimension along which the sequence advances: axis 0 advances coordinate index 0, axis 1 advances coordinate index 1.

## Validity rules
- The submitted sequence must be accepted by the formal language.
- Existing cells may be reused only with their existing symbol.
- Reuse at least one existing cell and place at least one new symbol.
- Only newly placed symbols consume the rack, including multiplicities.
- Do not reuse a cell whose existing word already runs along the chosen axis.
- The cell immediately before and after the submitted sequence on its axis must not continue an existing word.
- A newly placed cell must not extend an already-valid word on any perpendicular axis.
- After placement, every maximal contiguous line of length greater than one that touches the move, on every axis, must be accepted by the formal language.

## Scoring
- Each alphabet symbol has a point value, listed below. A valid move scores the sum of the point values of every symbol in its placed sequence, counting both newly placed and reused (overlapping) symbols. Higher-value symbols and longer valid sequences score more.
- An invalid move scores 0.

Letter scores:
  A: 2
  H: 1
  O: 2
  Q: 1
  R: 1
  S: 5
  U: 1
  Y: 1

Formal language:
Language ID: 1g_1b_lmh.low.r00.g00
Alphabet: {A, H, O, Q, R, S, U, Y}
k: 3
Minimum word length: 3
Forbidden snippets: {A A Y, A H H, A H O, A H U, A O H, A Q Q, A R H, A R S, A U A, A Y S, H A H, H A O, H H Y, H O H, H Q S, H U A, H U O, H Y H, O A O, O A Q, O A Y, O H Y, O O R, O Q Q, O R A, O S H, O Y U, O Y Y, Q A Y, Q H O, Q H Q, Q O O, Q Q H, Q Q R, Q R H, Q S R, Q U Y, Q Y R, Q Y Y, R A H, R A Q, R H Q, R H U, R O H, R O R, R O U, R Q O, R Q S, R Q U, R Q Y, R S A, R S Q, R S U, R S Y, R U H, R Y O, R Y S, R Y U, S H A, S H S, S H U, S O Y, S R A, S R R, S S R, S S Y, S U O, S U S, U A O, U A S, U A U, U A Y, U H S, U H Y, U Q R, U U R, U U S, U Y O, Y A A, Y H S, Y O U, Y Q H, Y Q Y, Y S O, Y U Q, Y Y A, Y Y O}
A sequence is valid iff it has the minimum length of 3 and contains no forbidden snippet.

Board configuration:
[omitted from notebook display: 80 occupied cells]

Rack:
["O", "S", "Y"]

```

### Raw model response
```text
{"start":[-8,1],"axis":0,"sequence":["S","Y","H","O"]}
```


## Response metrics

In [13]:
display(display_attempt_response(context))

## Evaluation summary

In [14]:
display(display_attempt_summary(context))

parse,OK
sequence,OK
spatial,OK
overlap,OK
no word extension,OK
cross words,OK
rack,OK
word length,4
overlap count,1
letter score,9


## Board visualization — parsed move

Blue = newly placed tile, green = reused tile that matches, red = symbol conflict.

In [15]:
for figure in plot_attempt_move(context, move_source="parsed"):
    display(figure)

## Board visualization — ground truth

Ground truth ≠ only valid solution; it is the move the generator witness-chain selected.

In [16]:
for figure in plot_attempt_move(context, move_source="ground_truth"):
    display(figure)